In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
from imports import *

### Load Configs

In [2]:
from config import main_config
from src.data_io import catalog
from src.utils.sessions import SESSION_COLUMNS

INCLUDE_EYE_TRACKING_SESSIONS = main_config.INCLUDE_EYE_TRACKING_SESSIONS

## Compiling data from all subjects

In [3]:
reject_subject = main_config["moca_rejection"]

In [4]:
subject_metadata = catalog.load("raw_session_metadata")
subjects_from_metadata = subject_metadata["subject_id"].unique()

# every session table compiled by 0.00 and 0.01, in name order so the joined table's row order is fixed
compiled_data_files = sorted(catalog.files("compiled_session", filename="*.parquet"))
subjects_from_data_file = [f.stem.split("_")[0] for f in compiled_data_files]

In [5]:
ucla = ["CG", "COH", "MBY", "DP", "FUR", "LBR", "MAR", "SMI", "PAM", "RW", "BBK", "BER", "DCAM", "ALE", "DMO", "AJL", "SKU"]
ucla_asmHC_EM = ["CB1", "JW", "KR", "MEG"]
ucla_asmHC_KP = ["AV", "BC", "BF", "EM", "ES", "GF", "GP", "JA", "MRM", "SY"]

case_western = ["RBA", "RDE", "SGA", "LHO", "RSH", "RZA", "SNO"]

stanford = subject_metadata.loc[
    subject_metadata["subject_id"].str.match(r"^P\d+$", na=False)  # Ensure it starts with 'P' followed by numbers
    & (subject_metadata["subject_id"].str.extract(r"P(\d+)")[0].astype(float) <= 24),  # Extract number and compare
    "subject_id",
].unique()
harvard = subject_metadata.loc[
    subject_metadata["subject_id"].str.match(r"^P\d+$", na=False)  # Ensure it starts with 'P' followed by numbers
    & (subject_metadata["subject_id"].str.extract(r"P(\d+)")[0].astype(float) > 24),  # Extract number and compare
    "subject_id",
].unique()

harvard_hc = subject_metadata.loc[subject_metadata["subject_id"].str.match(r"^HC\d+$", na=False), "subject_id"].unique()

# go through the subject_metadata and update the site
for subject_id in subject_metadata["subject_id"]:
    if subject_id in ucla or subject_id in ucla_asmHC_EM or subject_id in ucla_asmHC_KP:
        site = "UCLA"
    elif subject_id in case_western:
        site = "Case_Western"
    elif subject_id in stanford:
        site = "Stanford"
    elif subject_id in harvard or subject_id in harvard_hc:
        site = "Harvard"
    else:
        site = "Unknown"
    subject_metadata.loc[subject_metadata["subject_id"] == subject_id, "experiment_site"] = site

In [6]:
# remove rejected subjects from subjects_from_metadata and subjects_from_data_file
subjects_from_metadata = [s for s in subjects_from_metadata if s not in reject_subject]
subjects_from_data_file = [s for s in subjects_from_data_file if s not in reject_subject]

In [7]:
print(f"Metadata contains extra: {set(subjects_from_metadata) - set(subjects_from_data_file)}")
print(f"Data files contain extra: {set(subjects_from_data_file) - set(subjects_from_metadata)}")
print(len(set(subjects_from_data_file)), len(set(subjects_from_metadata)))

# Only assert if *all* subject IDs follow the pattern (none outside it)
if len(subjects_from_metadata) == len(subjects_from_metadata) and len(subjects_from_data_file) == len(subjects_from_data_file):
    assert len(set(subjects_from_data_file)) == len(set(subjects_from_metadata)), "Mismatch between metadata and data files"
else:
    pass

Metadata contains extra: set()
Data files contain extra: set()
74 74


In [8]:
# eye-tracking sessions (Perugini EM) are left out unless the config includes them
asmHC_EM_files = [f for f in compiled_data_files if f.stem.split("_")[0] in ucla_asmHC_EM]
session_files = [f for f in compiled_data_files if INCLUDE_EYE_TRACKING_SESSIONS or f not in asmHC_EM_files]
print(f"session tables: {len(compiled_data_files)}, eye-tracking: {len(asmHC_EM_files)}, joined: {len(session_files)}")

session tables: 151, eye-tracking: 4, joined: 147


In [9]:
aggregate_df = pd.concat([catalog.read(f) for f in session_files], ignore_index=True)

if not INCLUDE_EYE_TRACKING_SESSIONS:
    # remove rows from metadata corresponding to eye-tracking sessions
    eye_tracking_subjects = [f.stem.split("_")[0] for f in asmHC_EM_files]
    subject_metadata = subject_metadata[~subject_metadata["subject_id"].isin(eye_tracking_subjects)]

# replace empty strings with NaN
aggregate_df.replace("", np.nan, inplace=True)

In [10]:
aggregate_df["subject_id"].unique()

array(['AJL', 'ALE', 'AV', 'BBK', 'BC', 'BER', 'BF', 'CG', 'COH', 'DCAM',
       'DMO', 'DP', 'EM', 'ES', 'FUR', 'GF', 'GP', 'HC12', 'HC13', 'HC1',
       'HC2', 'HC3', 'HC5', 'HC6', 'HC7', 'HC8', 'HC9', 'JA', 'LBR',
       'LHO', 'MAR', 'MBY', 'MRM', 'P10', 'P11', 'P12', 'P13', 'P14',
       'P15', 'P16', 'P17', 'P18', 'P19', 'P1', 'P20', 'P22', 'P23',
       'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P2', 'P30', 'P31',
       'P32', 'P33', 'P34', 'P3', 'P4', 'P6', 'P7', 'P8', 'P9', 'PAM',
       'RBA', 'RDE', 'RSH', 'RW', 'RZA', 'SGA', 'SKU', 'SMI', 'SNO', 'SY'],
      dtype=object)

In [11]:
aggregate_df = aggregate_df[SESSION_COLUMNS]

In [12]:
catalog.save(aggregate_df, "compiled_trials")
catalog.save(subject_metadata, "compiled_metadata")

saved compiled_trials -> /mnt/pd-data/compiled/aggregate_all_data.csv
saved compiled_metadata -> /mnt/pd-data/compiled/aggregate_all_metadata.csv
